No,5

In [1]:
from pathlib import Path


# --- 共通の復元基盤 undo_utils.py を読み込む ---
# ※ このセルの import を並べ替えても壊れないように、undo_utils だけは
#    import 文ではなく importlib 経由で読み込んでいます。
import importlib
import sys
from pathlib import Path


def _locate_undo_utils():
    """undo_utils.py があるフォルダを探す（VS Code の作業ディレクトリ設定に依存しない）。"""
    candidates = []
    # 1) VS Code がノートブック自身のパスを教えてくれる場合
    nb_file = globals().get("__vsc_ipynb_file__")
    if nb_file:
        candidates.append(Path(nb_file).parent)
    # 2) 作業ディレクトリと、その中／親の「コードフォルダ」
    cwd = Path.cwd()
    candidates += [cwd, cwd / "コードフォルダ", cwd.parent, cwd.parent / "コードフォルダ"]

    for c in candidates:
        if (c / "undo_utils.py").is_file():
            return c.resolve()

    raise FileNotFoundError(
        "undo_utils.py が見つかりません。\n"
        "このノートブックと同じ「コードフォルダ」内に undo_utils.py があるか確認してください。\n"
        f"探した場所: {[str(c) for c in candidates]}"
    )


_uu_dir = str(_locate_undo_utils())
if _uu_dir not in sys.path:
    sys.path.insert(0, _uu_dir)

uu = importlib.import_module("undo_utils")
importlib.reload(uu)  # undo_utils.py を編集した場合も反映されるようにする

<module 'undo_utils' from 'C:\\Users\\0uh2j\\Desktop\\vscodeで\\ファイル整理２\\コードフォルダ\\undo_utils.py'>

ファイル名の末尾にそのファイルが入っているひとつ前のフォルダ名を付け足す。

NRとFutaba兼用

In [2]:
# === ファイル名の末尾に、ひとつ前のフォルダ名を付け足す ===
# 変更内容は _undo/undo_log.json に記録され、末尾の復元セルで元に戻せます。

STEP_NAME = "05_ファイル名の末尾にフォルダ名を付加"


def add_parent_folder_name():
    root_dir = uu.select_folder("対象のおおもとのフォルダを選択してください")
    if root_dir is None:
        return

    rename_count = 0
    print("--- 処理開始 ---")

    with uu.UndoJournal(root_dir, STEP_NAME) as j:
        # uu.iter_files は _undo / _trash と隠しファイルを除外して再帰的に走査する
        for file_path in list(uu.iter_files(root_dir)):
            # 末端からひとつ前のフォルダ
            parent_folder = file_path.parent.parent

            # 階層が浅すぎる（おおもとより外側を参照してしまう）場合はスキップ
            if not (root_dir in parent_folder.parents or root_dir == parent_folder):
                continue

            parent_name = parent_folder.name
            new_file_name = f"{file_path.stem}_{parent_name}{file_path.suffix}"
            new_file_path = file_path.with_name(new_file_name)

            if file_path.name == new_file_name:
                continue  # すでに同じ名前になっている

            if new_file_path.exists():
                print(f"スキップ: {new_file_name} はすでに存在します。")
                continue

            j.move(file_path, new_file_path)
            print(f"変更しました: {file_path.name}  ->  {new_file_name}")
            rename_count += 1

    print("--- 処理完了 ---")
    print(f"合計 {rename_count} 件のファイル名を変更しました。")


add_parent_folder_name()

選択されたフォルダ: C:/Users/0uh2j/Desktop/実験データ2026/01_本実験データ/202603-4 - 本実験圧力データ/202606・08-実験データ再々/202606・08-間接式樹脂圧力本実験データ再々/csv
--- 処理開始 ---
変更しました: 001_190℃_020.csv  ->  001_190℃_020_0.5mm_futaba.csv
変更しました: 002_190℃_020.csv  ->  002_190℃_020_0.5mm_futaba.csv
変更しました: 003_190℃_020.csv  ->  003_190℃_020_0.5mm_futaba.csv
変更しました: 004_190℃_020.csv  ->  004_190℃_020_0.5mm_futaba.csv
変更しました: 005_190℃_020.csv  ->  005_190℃_020_0.5mm_futaba.csv
変更しました: 006_190℃_020.csv  ->  006_190℃_020_0.5mm_futaba.csv
変更しました: 007_190℃_020.csv  ->  007_190℃_020_0.5mm_futaba.csv
変更しました: 008_190℃_020.csv  ->  008_190℃_020_0.5mm_futaba.csv
変更しました: 009_190℃_020.csv  ->  009_190℃_020_0.5mm_futaba.csv
変更しました: 010_190℃_020.csv  ->  010_190℃_020_0.5mm_futaba.csv
変更しました: 011_190℃_040.csv  ->  011_190℃_040_0.5mm_futaba.csv
変更しました: 012_190℃_040.csv  ->  012_190℃_040_0.5mm_futaba.csv
変更しました: 013_190℃_040.csv  ->  013_190℃_040_0.5mm_futaba.csv
変更しました: 014_190℃_040.csv  ->  014_190℃_040_0.5mm_futaba.csv
変更しました: 015_190℃_040.csv 

---
### ⏪ 復元（元に戻す）

このセルを実行すると、**このノートブックで行った直前の1工程**を巻き戻します。
（メインフォルダの `_undo/undo_log.json` に記録された履歴を使います）

繰り返し実行すれば、01〜06 のどの工程まででもさかのぼれます。
削除したファイルは `_trash` フォルダに退避されているので、これも一緒に元の場所へ戻ります。

In [ ]:
# ===== 共通の復元セル =====
# 直前に実行した1工程を巻き戻します。
# 続けて実行すれば、さらに1つ前の工程へとさかのぼれます。

uu.undo_interactive()